# Learning a quadratic pseudo-metric from distance measurements

Recall that pseudo-metric is a generalization of a metric space in which the distance between two distinct points can be zero.
We are given a set of $N$ pairs of points in $\mathbf{R}^n$, $x_1, \ldots, x_N$, and $y_1, \ldots, y_N$, together with a set of distances $d_1, \ldots, d_N > 0$.
  The goal is to find (or estimate or learn) a quadratic pseudo-metric $d$
  $$d(x,y) =  \left( (x-y)^T P(x-y) \right)^{1/2},$$
  $P\in \mathbf{S}^n_{+}$, which approximates the given distances, i.e., $d(x_i, y_i) \approx d_i$. (The pseudo-metric $d$ is a metric only when $P \succ 0$; when $P\succeq 0$ is singular, it is a pseudo-metric.)
  
  To do this, we will choose $P\in \mathbf{S}^n_+$ that minimizes the mean squared error objective
  
  $$f(S)=\frac{1}{N}\sum_{i=1}^N (d_i - d(x_i,y_i))^2.$$
  
  ### Theoretical part.
  1. Show that the objective function $f$ is convex (Hint: expand the square and see what happens.)
  2. Show that the convex program $\text{minimize }f(S)$, $S\succeq 0$ can be expressed by an equivalent conic program with linear objective and a number of conic constraints using the $R^n_+$ (nonnegative orthant cone), $Q^n$ (second order cone), $Q_r^n$ (rotated second order cone), $S^n_+$ (positive semidefinite cone).
  
  ### Programming Part
  1. Solve the program $\text{minimize }f(S)$, $S\succeq 0$, preferably using a modelling package like ``cvxpy``. Note that "under the hood" your modelling package translates the program to the conic form in point 2. above.
  2. Use the obtained $P$ to measure the mean square error for the test data ``X_test``, ``Y_test``, ``d_test``.
  
---- 
*This exercise originates from "Additional Exercises" collection for Convex Optimization textbook of S. Boyd and L. Vandenberghe. Used under permission*

In [1]:
import cvxpy as cp
import numpy as np
from scipy import linalg as la

In [2]:
# In this box we generate the input data

np.random.seed(5680)

n = 5 # Dimension
N = 100 # Number of samples

P = np.random.randn(n,n)
P = P.dot(P.T) + np.identity(n)
sqrtP = la.sqrtm(P)

x = np.random.randn(N,n)
y = np.random.randn(N,n)

d = np.linalg.norm(sqrtP.dot((x-y).T),axis=0)    # distances according to metric P
d = np.maximum(d+np.random.randn(N),0)           # add random noise

N_test = 10 # Samples for test set
X_test = np.random.randn(N_test,n)
Y_test = np.random.randn(N_test,n)
d_test = np.linalg.norm(sqrtP.dot((X_test-Y_test).T),axis=0)  # distances according to metric P
d_test = np.maximum(d_test+np.random.randn(N_test),0)         # add random noise


## Theoretical part

Let $z_i = x_i - y_i$ and $S \in \mathbf{S}^n_+$ be the unknown matrix.
Then $d(x_i, y_i) = \sqrt{z_i^T S z_i}$ and

$$
(d_i - d(x_i, y_i))^2 = d_i^2 - 2 d_i \sqrt{z_i^T S z_i} + z_i^T S z_i.
$$

### 1. Convexity of $f$

The objective is a sum of three terms (per sample $i$):

- $d_i^2$ is a constant.
- $z_i^T S z_i = \operatorname{tr}(S \, z_i z_i^T)$ is **linear** (hence convex) in $S$.
- $-2 d_i \sqrt{z_i^T S z_i}$: the argument $z_i^T S z_i$ is linear and nonnegative on $S \succeq 0$, and $\sqrt{\cdot}$ is concave and nondecreasing on $\mathbb{R}_+$, so $\sqrt{z_i^T S z_i}$ is concave in $S$. Since $d_i \ge 0$, multiplying by $-2 d_i$ makes the term **convex** in $S$.

A nonnegative sum of convex functions is convex, so $f$ is convex on $\mathbf{S}^n_+$.

### 2. Conic reformulation

Introduce auxiliary variables $t \in \mathbb{R}^N$ and $s \in \mathbb{R}^N$ and consider

$$
\begin{aligned}
\text{minimize}\quad & \frac{1}{N}\sum_{i=1}^N \big(d_i^2 - 2 d_i t_i + s_i\big) \\
\text{subject to}\quad
& s_i = z_i^T S z_i, \qquad i = 1, \dots, N, \\
& t_i^2 \le s_i, \quad t_i \ge 0,  \qquad i = 1, \dots, N, \\
& S \succeq 0.
\end{aligned}
$$

The objective is **linear** in $(S, t, s)$. The constraints use:

- The PSD cone $\mathbf{S}^n_+$ for $S \succeq 0$.
- The nonnegative orthant $\mathbb{R}^N_+$ for $t_i \ge 0$ (and the implicit $s_i \ge 0$).
- The rotated second-order cone $Q_r^3$ for each $t_i^2 \le s_i$, namely $(t_i,\, \tfrac{1}{2},\, s_i) \in Q_r^3$ (since $u^2 \le 2 v w$ with $v = 1/2,\, w = s_i$ gives $u^2 \le s_i$).
- Linear equalities $s_i = \operatorname{tr}(S\, z_i z_i^T)$.

Since $d_i \ge 0$, at the optimum each rotated-SOC constraint is tight, giving $t_i = \sqrt{z_i^T S z_i} = d(x_i, y_i)$, so this conic program is equivalent to minimizing the original $f(S)$. (The second-order cone $Q^n$ is not needed here; only $\mathbb{R}^N_+$, $Q_r^3$, and $\mathbf{S}^n_+$ are used.)

## Programming part

### 1. Solve $\min f(S)$ subject to $S \succeq 0$ with `cvxpy`

We model the conic reformulation above. The objective
$\tfrac{1}{N}\sum_i (z_i^T S z_i - 2 d_i t_i + d_i^2)$ is linear in the
variables $(S, t)$; the constraint $t_i^2 \le z_i^T S z_i$ is recognized by
cvxpy as a rotated second-order cone constraint.

In [ ]:
Z = x - y  # shape (N, n);  Z[i] = x_i - y_i

S = cp.Variable((n, n), symmetric=True)
t = cp.Variable(N, nonneg=True)

# quad_i = z_i^T S z_i, expressed as a linear function of S via trace(z z^T S)
quad = cp.hstack([cp.trace(np.outer(Z[i], Z[i]) @ S) for i in range(N)])

constraints = [S >> 0, cp.square(t) <= quad]

objective = cp.Minimize(cp.sum(quad - 2 * cp.multiply(d, t) + d ** 2) / N)

prob = cp.Problem(objective, constraints)
prob.solve()

P_hat = S.value
print("solver status:", prob.status)
print("training MSE f(P_hat) =", prob.value)
print("recovered P_hat =\n", P_hat)

### 2. Test MSE

Using the learned $\hat P$, evaluate
$\mathrm{MSE}_{\text{test}} = \frac{1}{N_{\text{test}}}\sum_i \big(d_{\text{test},i} - \sqrt{(X_{\text{test},i}-Y_{\text{test},i})^T \hat P (X_{\text{test},i}-Y_{\text{test},i})}\big)^2.$

In [ ]:
Z_test = X_test - Y_test
d_hat_test = np.sqrt(np.einsum("ij,jk,ik->i", Z_test, P_hat, Z_test))
test_mse = np.mean((d_test - d_hat_test) ** 2)
print("test MSE =", test_mse)

# For reference, the MSE of the *true* generating P on the same data:
d_hat_true = np.sqrt(np.einsum("ij,jk,ik->i", Z_test, P, Z_test))
print("test MSE with true P (sanity check) =",
      np.mean((d_test - d_hat_true) ** 2))